# Multimodal Dental Disease Diagnosis
## End-to-End PyTorch Pipeline with 5-Fold Cross-Validation
### CNN is INSIDE the fusion model — Grad-CAM gradients flow from fused logits to voxels

## Section 1: Config

In [1]:
# ============================================================
# SECTION 1: CONFIG
# ============================================================
import subprocess
subprocess.run(["pip","install","monai","-q"], check=True)
import os, re, gc, random, logging, warnings, time, html, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import monai
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import SimpleITK as sitk
from scipy.ndimage import zoom, rotate, gaussian_filter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score, accuracy_score,
    balanced_accuracy_score, fbeta_score)
from sklearn.linear_model import LogisticRegression
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

# ── disease labels ──
DISEASE_ORDER = ["pulpitis","caries","impacted_tooth","damaged_or_missing_tooth"]
NUM_CLASSES   = len(DISEASE_ORDER)

# ── modality dimensions ──
HIDDEN_SIZE   = 768    # ClinicalModernBERT hidden dim
EMBED_DIM     = 256    # shared encoder output dim
ATTN_DIM      = 64     # cross-attention head dim
N_FIELDS      = 5      # number of text fields
INPUT_SHAPE   = (96, 96, 96)  # preprocessed CBCT shape
HU_MIN, HU_MAX = -500.0, 2500.0

# ── model ──
DROPOUT       = 0.5
HEAD_DIM      = 64     # classification head hidden dim

# ── training ──
BATCH_SIZE    = 4      # CBCT volumes per batch (memory-aware)
EPOCHS        = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-3
PATIENCE      = 10
AUX_WEIGHT    = 0.5    # CBCT auxiliary loss weight
FREEZE_LAYERS = ("conv1","bn1","layer1","layer2")  # frozen MedicalNet layers

# ── data ──
BERT_MODEL_NAME    = "Simonlee711/Clinical_ModernBERT"
MEDICALNET_WEIGHTS = "/kaggle/working/medicalnet_resnet10_23.pth"
TEXT_FIELDS = ["main_appeal","subsequent","present_medical_history",
               "past_medical_history","oral_check"]

print(f"PyTorch {torch.__version__} | device: {device}")
print(f"Classes: {DISEASE_ORDER}")
print(f"CBCT input shape: {INPUT_SHAPE} | EMBED_DIM: {EMBED_DIM}")


PyTorch 2.10.0+cu128 | device: cuda
Classes: ['pulpitis', 'caries', 'impacted_tooth', 'damaged_or_missing_tooth']
CBCT input shape: (96, 96, 96) | EMBED_DIM: 256


## Section 2: MedicalNet Weights

In [2]:
# ============================================================
# SECTION 2: DOWNLOAD MEDICALNET WEIGHTS
# ============================================================
if not Path(MEDICALNET_WEIGHTS).exists():
    from huggingface_hub import hf_hub_download
    import shutil
    src = hf_hub_download(repo_id="TencentMedicalNet/MedicalNet-Resnet10",
                          filename="resnet_10_23dataset.pth")
    shutil.copy(src, MEDICALNET_WEIGHTS)
    print("downloaded MedicalNet weights")
print("weights present:", Path(MEDICALNET_WEIGHTS).exists())


weights present: True


## Section 3: Data Loading

In [3]:
# ============================================================# SECTION 3: DATA LOADING (adapted for manifest.csv + mmdental-dataset)# ============================================================CSV_PATH = list(Path("/kaggle/input").rglob("manifest.csv"))if not CSV_PATH:    raise FileNotFoundError("manifest.csv not found. Attach mmdental-essentials dataset.")CSV_PATH = CSV_PATH[0]print(f"Manifest: {CSV_PATH}")raw_df = pd.read_csv(CSV_PATH, low_memory=False)raw_df.columns = [c.strip().lower().replace(" ","_") for c in raw_df.columns]raw_df["patient_id"] = raw_df["filename"].astype(str).str.strip()print(f"Patients: {len(raw_df)}")# Parse the pipe-delimited 'text' column into individual clinical fields.# Format: "[Main appeal] ... || [Present medical history] ... || [Oral Check] ..."import redef parse_text_fields(text_str):    fields = {f: "" for f in TEXT_FIELDS}    if not isinstance(text_str, str) or text_str.strip() == "":        return fields    # Split on || and match [field_name] content    parts = re.split(r'\|\|', text_str)    field_map = {        "main appeal": "main_appeal",        "present medical history": "present_medical_history",        "past medical history": "subsequent",        "oral check": "diagnosis",        "diagnosis": "diagnosis",        "treatment": "subsequent",    }    for part in parts:        part = part.strip()        m = re.match(r'\[([^\]]+)\]\s*(.*)', part, re.DOTALL)        if m:            key = m.group(1).strip().lower()            val = m.group(2).strip()            mapped = field_map.get(key)            if mapped:                if fields[mapped]:                    fields[mapped] += " | " + val                else:                    fields[mapped] = val    return fields# Build patient_df: one row per patient with age, sex, text fields, cbct path, labelspatient_rows = []for _, r in raw_df.iterrows():    row = {}    row["patient_id"] = r["patient_id"]    row["age"] = r["age"] if 1 < r["age"] < 120 else np.nan    row["sex"] = str(r["sex"]).strip().lower() if pd.notna(r["sex"]) and str(r["sex"]).strip().lower() in ("male","female") else np.nan    # Parse clinical text    text_fields = parse_text_fields(r.get("text", ""))    for f in TEXT_FIELDS:        row[f] = text_fields.get(f, "")    row["diagnosis"] = text_fields.get("diagnosis", "")    row["main_appeal"] = text_fields.get("main_appeal", "")    # CBCT path from volume_path column    row["cbct_path"] = r.get("volume_path") if pd.notna(r.get("volume_path")) else None    # Pre-computed labels from manifest (0/1 → our DISEASE_ORDER)    manifest_to_disease = {        "caries": "caries",        "pulpitis": "pulpitis",        "impacted_tooth": "impacted_tooth",        "missing_damaged": "damaged_or_missing_tooth",    }    for mcol, dcol in manifest_to_disease.items():        row[f"label_{dcol}"] = int(r.get(mcol, 0)) if pd.notna(r.get(mcol)) else 0    patient_rows.append(row)patient_df = pd.DataFrame(patient_rows)patient_df["cbct_valid"] = patient_df["cbct_path"].apply(    lambda p: p is not None and isinstance(p, str) and Path(p).exists())patient_df.loc[~patient_df["cbct_valid"], "cbct_path"] = Nonepatient_df["cbct_present"] = patient_df["cbct_valid"].astype(np.float32)patient_df["structured_present"] = (patient_df["age"].notna() & patient_df["sex"].notna()).astype(np.float32)patient_df["text_present"] = (patient_df[TEXT_FIELDS].apply(lambda c: c.str.strip().str.len() > 0).any(axis=1)).astype(np.float32)print(f"Valid CBCT: {int(patient_df['cbct_present'].sum())} | structured: {int(patient_df['structured_present'].sum())} | text: {int(patient_df['text_present'].sum())}")# ── pre-computed labels (from manifest) ──label_matrix = np.stack([    patient_df[f"label_{d}"].values.astype(np.float32)    for d in DISEASE_ORDER], axis=1)# Keep patients with at least one label OR valid CBCTkeep = (label_matrix.sum(1) > 0) | (patient_df["cbct_present"].values > 0)patient_df = patient_df[keep].reset_index(drop=True)label_matrix = label_matrix[keep]print(f"After filtering: {len(patient_df)} patients")for i, d in enumerate(DISEASE_ORDER):    print(f"  {d:<26}{int(label_matrix[:, i].sum()):>4} ({label_matrix[:, i].mean()*100:.1f}%)")

CSV: /kaggle/input/datasets/lawrenciaasareansa/mmdental-dataset/MMDental/medical_records.csv | scans: 403
Patients: 660 | valid CBCT: 403


## Section 4: Labels (Core-4 ICD-anchored)

In [4]:
# ============================================================# SECTION 4: LABELS (from manifest.csv pre-computed labels)# ============================================================# Labels are already extracted from manifest.csv columns.# Verified: pulpitis, caries, impacted_tooth, missing_damaged map to DISEASE_ORDER.print(f"Label matrix shape: {label_matrix.shape}")print(f"Total positive labels: {int(label_matrix.sum())}")for i, d in enumerate(DISEASE_ORDER):    print(f"  {d:<26}{int(label_matrix[:, i].sum()):>4} ({label_matrix[:, i].mean()*100:.1f}%)")

Patients with >=1 Core-4: 459 | CBCT-present: 285
  pulpitis                   150 (32.7%)
  caries                     138 (30.1%)
  impacted_tooth             122 (26.6%)
  damaged_or_missing_tooth   218 (47.5%)


## Section 5: Structured Features

In [5]:
# ============================================================
# SECTION 5: STRUCTURED FEATURES
# Scaler fitted on the initial train split (for normalisation only).
# CV re-pools everything afterward.
# ============================================================
patient_df["sex_male"]  = (patient_df["sex"]=="male").astype(np.float32)
patient_df["sex_female"]= (patient_df["sex"]=="female").astype(np.float32)
idx    = np.arange(len(patient_df))
tr_i,te_i = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=label_matrix[:,3])
tr_i,va_i = train_test_split(tr_i, test_size=0.1765, random_state=SEED, stratify=label_matrix[tr_i,3])
train_df = patient_df.iloc[tr_i].reset_index(drop=True)
val_df   = patient_df.iloc[va_i].reset_index(drop=True)
test_df  = patient_df.iloc[te_i].reset_index(drop=True)
y_train, y_val, y_test = label_matrix[tr_i], label_matrix[va_i], label_matrix[te_i]

age_mean = train_df["age"].mean()
scaler   = StandardScaler()
scaler.fit(train_df["age"].fillna(age_mean).values.reshape(-1,1))
def build_struct(df):
    a = scaler.transform(df["age"].fillna(age_mean).values.reshape(-1,1))
    f = np.column_stack([a.flatten(),df["sex_male"].values,df["sex_female"].values]).astype(np.float32)
    p = df["structured_present"].values.astype(np.float32).reshape(-1,1)
    f[p.flatten()==0]=0.0; return f,p
struct_train,sp_train = build_struct(train_df)
struct_val,  sp_val   = build_struct(val_df)
struct_test, sp_test  = build_struct(test_df)
print(f"Structured — train: {struct_train.shape} | val: {struct_val.shape} | test: {struct_test.shape}")
print(f"Split — train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")


Structured — train: (321, 3) | val: (69, 3) | test: (69, 3)
Split — train 321 | val 69 | test 69


## Section 6: CBCT Preprocessing (offline, saved as volumes)

In [6]:
# ============================================================
# SECTION 6: CBCT PREPROCESSING
# Preprocessing is still done OFFLINE and saved — the CNN runs
# LIVE inside the fusion model at training time (end-to-end).
# ============================================================
def preprocess_cbct(path, augment=False):
    try:
        vol = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
        vol = np.nan_to_num(vol,nan=HU_MIN,posinf=HU_MAX,neginf=HU_MIN)
        vol = np.clip(vol,HU_MIN,HU_MAX)
        c   = np.argwhere(vol>-400.0)
        if c.size:
            z0,y0,x0=c.min(0); z1,y1,x1=c.max(0)
            vol=vol[z0:z1+1,y0:y1+1,x0:x1+1]
        vol = vol[:int(vol.shape[0]*0.65),:,:]
        vol = (vol-vol.min())/(vol.max()-vol.min()+1e-9)
        if augment:
            if np.random.rand()<0.5: vol=np.flip(vol,axis=2).copy()
            if np.random.rand()<0.7: vol=rotate(vol,np.random.uniform(-12,12),axes=(1,2),reshape=False,order=1,mode="constant",cval=0.0)
            if np.random.rand()<0.4: vol=rotate(vol,np.random.uniform(-7,7),axes=(0,1),reshape=False,order=1,mode="constant",cval=0.0)
            if np.random.rand()<0.5: vol=vol*np.random.uniform(0.9,1.1)+np.random.uniform(-0.05,0.05)
            if np.random.rand()<0.3: vol=gaussian_filter(vol,sigma=np.random.uniform(0.3,0.7))
            if np.random.rand()<0.3: vol=vol+np.random.normal(0,0.02,vol.shape).astype(np.float32)
            vol=np.clip(vol,0,1)
        zf  = [t/max(s,1) for t,s in zip(INPUT_SHAPE,vol.shape)]
        vol = zoom(vol,zf,order=1)
        vol = vol[:INPUT_SHAPE[0],:INPUT_SHAPE[1],:INPUT_SHAPE[2]]
        vol = np.pad(vol,[(0,INPUT_SHAPE[k]-vol.shape[k]) for k in range(3)],mode="constant")
        return vol[np.newaxis,...].astype(np.float32)   # (1,96,96,96)
    except Exception as e:
        print("prep fail",repr(e)[:60])
        return np.zeros((1,*INPUT_SHAPE),dtype=np.float32)

# preload ALL patients' preprocessed volumes (not just CBCT-present)
# CBCT-absent patients get a zero volume — the cbct_present flag masks them
print("Preprocessing CBCT volumes for all patients...")
all_vols = np.zeros((len(patient_df),1,*INPUT_SHAPE),dtype=np.float32)
for i,(_,r) in enumerate(patient_df.iterrows()):
    if pd.notna(r["cbct_path"]) and r["cbct_path"] is not None:
        all_vols[i] = preprocess_cbct(r["cbct_path"],augment=False)
    if i%50==0: print(f"  {i+1}/{len(patient_df)}",flush=True)
np.save(OUTPUT_DIR/"all_vols.npy", all_vols)
print(f"Saved all_vols.npy — shape: {all_vols.shape} | size: {all_vols.nbytes/1e9:.2f} GB")


Preprocessing CBCT volumes for all patients...
  1/459
  51/459
  101/459
  151/459
  201/459
  251/459
  301/459
  351/459
  401/459
  451/459
Saved all_vols.npy — shape: (459, 1, 96, 96, 96) | size: 1.62 GB


## Section 7: Text Embeddings (ClinicalModernBERT, offline)

In [7]:
# ============================================================
# SECTION 7: TEXT EMBEDDINGS — still cached offline
# Text embedding is not end-to-end (BERT frozen, features cached).
# Only the CBCT branch is end-to-end.
# ============================================================
print("Loading ClinicalModernBERT...")
tokenizer   = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model  = AutoModel.from_pretrained(BERT_MODEL_NAME).eval()
bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model  = bert_model.to(bert_device)

def embed_field(text, max_len=512):
    if not isinstance(text,str) or text.strip()=="":
        return np.zeros(HIDDEN_SIZE,dtype=np.float32)
    enc  = tokenizer(text,truncation=True,max_length=max_len,return_tensors="pt")
    ids  = enc["input_ids"].to(bert_device)
    mask = enc["attention_mask"].to(bert_device)
    with torch.no_grad():
        out = bert_model(input_ids=ids,attention_mask=mask,output_hidden_states=True)
    last4 = torch.stack(out.hidden_states[-4:],0).mean(0)
    m     = mask.unsqueeze(-1).float()
    emb   = (last4*m).sum(1)/m.sum(1).clamp(min=1e-9)
    return emb.squeeze(0).cpu().numpy().astype(np.float32)

def extract_fields(df):
    E=np.zeros((len(df),N_FIELDS,HIDDEN_SIZE),dtype=np.float32)
    P=np.zeros((len(df),1),dtype=np.float32)
    for i,(_,row) in enumerate(df.iterrows()):
        embs=[embed_field(str(row[f])) if pd.notna(row[f]) else np.zeros(HIDDEN_SIZE,dtype=np.float32) for f in TEXT_FIELDS]
        present=any(pd.notna(row[t]) and str(row[t]).strip() for t in TEXT_FIELDS)
        E[i]=np.stack(embs); P[i,0]=1.0 if present else 0.0
    E[P.flatten()==0]=0.0; return E,P

# extract for ALL patients (same split-independent pool the CV will use)
all_field_emb, all_text_pres = extract_fields(patient_df)
print(f"Text embeddings: {all_field_emb.shape}")

# also extract split-specific (for the standalone test-set eval after CV)
field_train,tp_train = extract_fields(train_df)
field_val,  tp_val   = extract_fields(val_df)
field_test, tp_test  = extract_fields(test_df)

# free BERT — no longer needed
del bert_model; gc.collect(); torch.cuda.empty_cache()
print("BERT freed.")


Loading ClinicalModernBERT...


Loading weights:   0%|          | 0/357 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Simonlee711/Clinical_ModernBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Text embeddings: (459, 5, 768)
BERT freed.


## Section 8: Structured Features for All Patients

In [8]:
# ============================================================
# SECTION 8: STRUCTURED FEATURES FOR ALL 459 PATIENTS
# (CV re-pools all patients, so we need struct for everyone)
# ============================================================
all_struct, all_sp = build_struct(patient_df)
all_cbct_pres = patient_df["cbct_present"].values.astype(np.float32).reshape(-1,1)
all_labels    = label_matrix.copy()
print(f"Pooled: {len(patient_df)} patients")
print(f"  struct: {all_struct.shape} | text: {all_field_emb.shape} | vols: {all_vols.shape}")


Pooled: 459 patients
  struct: (459, 3) | text: (459, 5, 768) | vols: (459, 1, 96, 96, 96)


## Section 9: End-to-End PyTorch Fusion Model

In [9]:
# ============================================================
# SECTION 9: END-TO-END PYTORCH FUSION MODEL
# LightMamba3D — pure PyTorch SSM (no mamba-ssm package needed)
# MambaMIM pretrained weights loaded where architecture matches.
# ============================================================
from huggingface_hub import hf_hub_download
import shutil
import math
CBCT_FEAT_DIM = 256   # output dimension of the CBCT encoder
# ── download MambaMIM pretrained weights ──
MAMBAMIM_PATH = "/kaggle/working/mambamim_mask75.pth"
if not Path(MAMBAMIM_PATH).exists():
    src = hf_hub_download(repo_id="FengheTan9/MambaMIM",
                          filename="mambamim_mask75.pth")
    shutil.copy(src, MAMBAMIM_PATH)
    print("MambaMIM weights downloaded")
print("MambaMIM weights present:", Path(MAMBAMIM_PATH).exists())

# ── 9a: pure PyTorch SSM (selective state space model) ──
class PurePyTorchSSM(nn.Module):
    """
    Simplified selective state space model — pure PyTorch, no mamba-ssm.
    Implements the core selective scan mechanism:
      x (B,L,D) → project to z,x → SSM scan → gate with z → output
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model  = d_model
        self.d_state  = d_state
        self.d_inner  = int(expand * d_model)

        # input projections
        self.in_proj  = nn.Linear(d_model, self.d_inner*2, bias=False)
        # depthwise conv for local context
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner,
                                  kernel_size=d_conv, padding=d_conv-1,
                                  groups=self.d_inner, bias=True)
        # SSM parameters
        self.x_proj   = nn.Linear(self.d_inner, d_state*2 + self.d_inner, bias=False)
        self.dt_proj  = nn.Linear(self.d_inner, self.d_inner, bias=True)
        # learnable A (state transition matrix, log-parameterised)
        A = torch.arange(1, d_state+1, dtype=torch.float).unsqueeze(0).repeat(self.d_inner,1)
        self.A_log    = nn.Parameter(torch.log(A))
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        # x: (B, L, D)
        B, L, D = x.shape
        xz   = self.in_proj(x)                        # (B,L,2*d_inner)
        x_, z = xz.chunk(2, dim=-1)                   # each (B,L,d_inner)

        # depthwise conv along sequence
        x_ = x_.transpose(1,2)                        # (B,d_inner,L)
        x_ = self.conv1d(x_)[..., :L]                 # causal: trim
        x_ = x_.transpose(1,2)                        # (B,L,d_inner)
        x_ = F.silu(x_)

        # selective SSM
       
        xBC  = self.x_proj(x_)                        # (B,L,d_state*2+d_inner)
        
        # 1. Split xBC into all 3 parts: dt_raw (256), B (16), and C (16)
        dt_raw, B_mat, C_mat = xBC.split([self.d_inner, self.d_state, self.d_state], dim=-1)
        
        # 2. Project the extracted dt_raw, NOT the original x_
        dt   = self.dt_proj(dt_raw)                   # (B,L,d_inner)
        dt   = F.softplus(dt)
        # discretize A
        A    = -torch.exp(self.A_log)                 # (d_inner, d_state)
        dA   = torch.exp(dt.unsqueeze(-1) * A)        # (B,L,d_inner,d_state)
        dB   = dt.unsqueeze(-1) * B_mat.unsqueeze(2)  # (B,L,d_inner,d_state)

        # parallel scan (simplified: sequential for correctness on small L)
        # at 12^3=1728 tokens this is fast enough on GPU
        h    = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys   = []
        for i in range(L):
            # Extract step i and add an empty dim at the end to make it (B, d_inner, 1)
            h = dA[:,i]*h + dB[:,i]*x_[:, i, :].unsqueeze(-1)
            y = (h * C_mat[:,i,:].unsqueeze(1)).sum(-1)  # (B,d_inner)
            ys.append(y)
        y    = torch.stack(ys, dim=1)                 # (B,L,d_inner)
        y    = y + x_ * self.D                        # skip connection
        y    = y * F.silu(z)                          # gating
        return self.out_proj(y)                        # (B,L,D)

class MambaBlock3D(nn.Module):
    """3D Mamba block — scans flattened spatial sequence with pure PyTorch SSM."""
    def __init__(self, dim, d_state=16, d_conv=4, expand=2, dropout=0.3):
        super().__init__()
        self.norm  = nn.LayerNorm(dim)
        self.ssm   = PurePyTorchSSM(dim, d_state=d_state,
                                     d_conv=d_conv, expand=expand)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        B,C,D,H,W = x.shape
        seq = x.flatten(2).transpose(1,2)              # (B, D*H*W, C)
        seq = seq + self.drop(self.ssm(self.norm(seq)))
        return seq.transpose(1,2).reshape(B,C,D,H,W)

# ── 9b: LightMamba3D encoder ──
class LightMamba3DEncoder(nn.Module):
    """
    Lightweight 3D Mamba encoder for CBCT volumes.
    96^3 → patch embed → 12^3 tokens → 3 Mamba blocks → GAP → 256-dim.
    Pure PyTorch: no external CUDA extensions needed.
    """
    def __init__(self, in_channels=1, embed_dim=128, n_blocks=3,
                 out_dim=256, dropout=0.3):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv3d(in_channels, embed_dim//2, kernel_size=4, stride=4),
            nn.GELU(),
            nn.Conv3d(embed_dim//2, embed_dim, kernel_size=2, stride=2),
            nn.GELU())
        self.blocks = nn.ModuleList([
            MambaBlock3D(embed_dim, d_state=16, d_conv=4,
                         expand=2, dropout=dropout)
            for _ in range(n_blocks)])
        self.norm = nn.LayerNorm(embed_dim)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.proj = nn.Sequential(nn.Dropout(dropout),
                                   nn.Linear(embed_dim, out_dim))
    def forward(self, x):
        x = self.patch_embed(x)
        for blk in self.blocks: x = blk(x)
        B,C,D,H,W = x.shape
        x = self.norm(x.flatten(2).transpose(1,2)).transpose(1,2).reshape(B,C,D,H,W)
        return self.proj(self.pool(x).flatten(1))

def build_mamba_encoder(pretrained_path=None, freeze_first_n_blocks=1):
    enc = LightMamba3DEncoder(in_channels=1, embed_dim=128, n_blocks=3,
                               out_dim=CBCT_FEAT_DIM, dropout=0.4)
    if pretrained_path and Path(pretrained_path).exists():
        ck     = torch.load(pretrained_path, map_location="cpu", weights_only=False)
        enc_sd = enc.state_dict(); loaded=0
        for k,v in ck.items():
            k2 = k.replace("encoder.","").replace("module.","")
            if k2 in enc_sd and enc_sd[k2].shape==v.shape:
                enc_sd[k2]=v; loaded+=1
        enc.load_state_dict(enc_sd, strict=False)
        print(f"MambaMIM partial transfer: {loaded} tensors matched")
    else:
        print("Training LightMamba3D from scratch")
    for i,blk in enumerate(enc.blocks):
        for p in blk.parameters():
            p.requires_grad = (i >= freeze_first_n_blocks)
    n_tr=sum(p.requires_grad for p in enc.parameters())
    n_to=sum(1 for _ in enc.parameters())
    print(f"LightMamba3DEncoder: {n_tr}/{n_to} param-tensors trainable")
    return enc.to(device)

# ── 9c: structured encoder ──
class StructuredEncoder(nn.Module):
    def __init__(self,in_dim=3,embed_dim=EMBED_DIM,dropout=DROPOUT):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim,128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128,embed_dim), nn.ReLU())
        self.miss = nn.Embedding(1,embed_dim)
    def forward(self,feats,pres):
        out  = self.mlp(feats)
        miss = self.miss(torch.zeros(feats.size(0),dtype=torch.long,device=feats.device))
        return out*pres + miss*(1-pres)

# ── 9d: text encoder with higher dropout ──
class TextEncoder(nn.Module):
    def __init__(self,hidden=HIDDEN_SIZE,embed_dim=EMBED_DIM,
                 attn_dim=ATTN_DIM,dropout=DROPOUT):
        super().__init__()
        self.proj  = nn.Linear(hidden,embed_dim)
        self.q_lin = nn.Linear(embed_dim,attn_dim)
        self.k_lin = nn.Linear(embed_dim,attn_dim)
        self.norm  = nn.LayerNorm(embed_dim)
        self.drop  = nn.Dropout(0.6)
        self.miss  = nn.Embedding(1,embed_dim)
    def forward(self,fields,pres):
        x   = torch.relu(self.proj(fields.float()))
        q   = x.mean(1,keepdim=True)
        k   = self.k_lin(x)
        qp  = self.q_lin(q)
        w   = torch.softmax(torch.bmm(qp,k.transpose(1,2))/
              (ATTN_DIM**0.5),dim=-1)
        txt = self.drop(self.norm((w@x).squeeze(1)))
        miss= self.miss(torch.zeros(fields.size(0),dtype=torch.long,device=fields.device))
        return txt*pres.float() + miss*(1-pres.float())

# ── 9e: CBCT encoder ──
class CBCTEncoder(nn.Module):
    def __init__(self,in_dim=256,embed_dim=EMBED_DIM,dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,256), nn.ReLU(), nn.LayerNorm(256),
            nn.Dropout(dropout), nn.Linear(256,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self,feat): return self.net(feat)

# ── 9f: classification head ──
class ClassHead(nn.Module):
    def __init__(self,in_dim=EMBED_DIM,n_classes=NUM_CLASSES,dropout=DROPOUT):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,HEAD_DIM),nn.ReLU(),
                               nn.Dropout(dropout),nn.Linear(HEAD_DIM,n_classes))
    def forward(self,x): return self.net(x)

# ── 9g: late fusion ──
class LateFusion(nn.Module):
    def __init__(self,n_mod=3,n_classes=NUM_CLASSES):
        super().__init__()
        self.w = nn.Parameter(torch.ones(n_mod,n_classes))
    def forward(self,st_l,tx_l,cb_l,sp,tp,cp):
        logits = torch.stack([st_l,tx_l,cb_l],dim=1)
        masks  = torch.cat([sp,tp,cp],dim=1).unsqueeze(-1)
        w      = torch.softmax(self.w,dim=0).unsqueeze(0)
        wm     = w*masks; wm=wm/(wm.sum(1,keepdim=True)+1e-6)
        return (wm*logits).sum(1)

# ── 9h: full end-to-end fusion model ──
class E2EFusionModel(nn.Module):
    """
    LightMamba3D encoder (pure PyTorch SSM) lives inside this model.
    Grad-CAM hooks cnn.blocks[-1] for honest attribution from fused logit.
    """
    def __init__(self, mamba_enc):
        super().__init__()
        self.cnn         = mamba_enc
        self.struct_enc  = StructuredEncoder()
        self.text_enc    = TextEncoder()
        self.cbct_enc    = CBCTEncoder()
        self.struct_head = ClassHead()
        self.text_head   = ClassHead()
        self.cbct_head   = ClassHead()
        self.fusion      = LateFusion()

    def forward(self, vol, cbct_pres, fields, text_pres, struct, struct_pres):
        B = vol.size(0)
        cnn_out  = torch.zeros(B,256,dtype=torch.float32,device=vol.device)
        has_cbct = (cbct_pres.squeeze(1)>0.5)
        if has_cbct.any():
            cnn_out[has_cbct] = self.cnn(vol[has_cbct]).float()
        cb_emb = self.cbct_enc(cnn_out)
        cb_l   = self.cbct_head(cb_emb)
        tx_emb = self.text_enc(fields, text_pres)
        tx_l   = self.text_head(tx_emb)
        st_emb = self.struct_enc(struct.float(), struct_pres.float())
        st_l   = self.struct_head(st_emb)
        fused  = self.fusion(st_l, tx_l, cb_l,
                             struct_pres.float(), text_pres.float(),
                             cbct_pres.float())
        return {"logits":fused, "cbct_aux_logits":cb_l,
                "text_logits":tx_l, "struct_logits":st_l}

# build
mamba_enc = build_mamba_encoder(MAMBAMIM_PATH, freeze_first_n_blocks=1)
model     = E2EFusionModel(mamba_enc).to(device)
n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nE2EFusionModel with LightMamba3D (pure PyTorch). Trainable: {n_params:,}")
print("No external CUDA extensions — runs on any Kaggle GPU.")
print("Grad-CAM: hook best_model.cnn.blocks[-1]")

MambaMIM weights present: True
MambaMIM partial transfer: 0 tensors matched
LightMamba3DEncoder: 30/41 param-tensors trainable

E2EFusionModel with LightMamba3D (pure PyTorch). Trainable: 1,037,592
No external CUDA extensions — runs on any Kaggle GPU.
Grad-CAM: hook best_model.cnn.blocks[-1]


## Section 10: Dataset and DataLoader

In [10]:
# ============================================================
# SECTION 10: PYTORCH DATASET
# Each item: preprocessed vol + text emb + struct + label
# Augmentation applied on-the-fly for CBCT-present patients
# ============================================================
class MultimodalDataset(Dataset):
    def __init__(self, vols, field_emb, text_pres, struct, struct_pres,
                 cbct_pres, labels, augment=False):
        self.vols        = vols          # (N,1,96,96,96)
        self.field_emb   = field_emb     # (N,5,768)
        self.text_pres   = text_pres     # (N,1)
        self.struct      = struct        # (N,3)
        self.struct_pres = struct_pres   # (N,1)
        self.cbct_pres   = cbct_pres     # (N,1)
        self.labels      = labels        # (N,4)
        self.augment     = augment

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        vol = self.vols[i].copy()   # (1,96,96,96)
        # augment only CBCT-present patients
        if self.augment and self.cbct_pres[i,0]>0.5:
            v = vol[0]
            if np.random.rand()<0.5: v=np.flip(v,axis=2).copy()
            if np.random.rand()<0.7: v=rotate(v,np.random.uniform(-12,12),axes=(1,2),reshape=False,order=1,mode="constant",cval=0.0)
            if np.random.rand()<0.4: v=rotate(v,np.random.uniform(-7,7),axes=(0,1),reshape=False,order=1,mode="constant",cval=0.0)
            if np.random.rand()<0.5: v=v*np.random.uniform(0.9,1.1)+np.random.uniform(-0.05,0.05)
            if np.random.rand()<0.3: v=gaussian_filter(v,sigma=np.random.uniform(0.3,0.7))
            if np.random.rand()<0.3: v=v+np.random.normal(0,0.02,v.shape).astype(np.float32)
            vol=np.clip(v,0,1)[np.newaxis]
        return {
            "vol":         torch.from_numpy(vol.astype(np.float32)),
            "fields":      torch.from_numpy(self.field_emb[i]),
            "text_pres":   torch.from_numpy(self.text_pres[i]),
            "struct":      torch.from_numpy(self.struct[i]),
            "struct_pres": torch.from_numpy(self.struct_pres[i]),
            "cbct_pres":   torch.from_numpy(self.cbct_pres[i]),
            "label":       torch.from_numpy(self.labels[i]),
        }

def make_loader(idx, augment, shuffle, batch_size=BATCH_SIZE):
    ds = MultimodalDataset(
        all_vols[idx], all_field_emb[idx], all_text_pres[idx],
        all_struct[idx], all_sp[idx], all_cbct_pres[idx],
        all_labels[idx], augment=augment)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

print("MultimodalDataset and make_loader ready")


MultimodalDataset and make_loader ready


## Section 11: Loss Functions and Helpers

In [11]:
# ============================================================
# SECTION 11: LOSS FUNCTIONS AND HELPERS
# ============================================================
def focal_loss(y_true, logits, alpha=0.25, gamma=2.0, reduction="mean"):
    """Multi-label focal loss."""
    p   = torch.sigmoid(logits)
    bce = F.binary_cross_entropy_with_logits(logits,y_true,reduction="none")
    p_t = y_true*p + (1-y_true)*(1-p)
    a_t = y_true*alpha + (1-y_true)*(1-alpha)
    fl  = a_t * (1-p_t)**gamma * bce
    return fl.mean() if reduction=="mean" else fl.mean(dim=1)

def aux_cbct_loss(y_true, cb_logits, cbct_pres):
    """Focal loss on CBCT branch, only for CBCT-present patients."""
    per  = focal_loss(y_true, cb_logits, reduction="none")
    mask = cbct_pres.squeeze(1)
    return (per*mask).sum()/(mask.sum()+1e-6)

def predict_probs_e2e(model, ds_idx, batch_size=8):
    """Run inference and return sigmoid probabilities."""
    model.eval()
    loader = make_loader(ds_idx, augment=False, shuffle=False, batch_size=batch_size)
    probs  = []
    with torch.no_grad():
        for batch in loader:
            out = model(batch["vol"].to(device), batch["cbct_pres"].to(device),
                        batch["fields"].to(device), batch["text_pres"].to(device),
                        batch["struct"].to(device), batch["struct_pres"].to(device))
            probs.append(torch.sigmoid(out["logits"]).cpu().numpy())
    return np.concatenate(probs,0)

def support_weighted_pr(y_true, probs):
    aps,sup=[],[]
    for c in range(NUM_CLASSES):
        s=y_true[:,c].sum()
        if 0<s<len(y_true): aps.append(average_precision_score(y_true[:,c],probs[:,c])); sup.append(s)
    return np.average(aps,weights=sup),aps,sup

def macro_pr(y,p): return average_precision_score(y,p,average="macro")
def macro_roc(y,p):
    v=[roc_auc_score(y[:,c],p[:,c]) for c in range(NUM_CLASSES) if 0<y[:,c].sum()<len(y)]
    return np.mean(v)

print("Loss functions and helpers ready.")


Loss functions and helpers ready.


## Section 12: 5-Fold Cross-Validation (End-to-End)

In [ ]:
# ============================================================
# SECTION 12: 5-FOLD CV — END-TO-END WITH MAMBA ENCODER
# Overfitting fixes:
# (1) Per-group weight decay: higher on text/struct (1e-2), lower on Mamba (1e-3)
# (2) Lower Mamba LR (1e-5) — fine-tune gently
# (3) Tighter early stopping: PATIENCE=8, EPOCHS=40
# ============================================================
from sklearn.model_selection import StratifiedKFold, train_test_split as _tts

skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
strat = all_labels[:,3]

fold_sw, fold_macro_pr, fold_macro_roc = [], [], []
per_class_ap = {d:[] for d in DISEASE_ORDER}
cv_store, cv_history = [], []
best_fold_metric, best_fold_id = -np.inf, -1
best_fold_data  = None
BEST_PATH       = str(OUTPUT_DIR/"best_e2e_fold.pt")

CV_EPOCHS   = 40    # tighter than original 60
CV_PATIENCE = 8     # tighter than original 10

for fold,(tr_idx,te_idx) in enumerate(skf.split(np.arange(len(all_labels)),strat)):
    tr2,va2 = _tts(tr_idx, test_size=0.15, random_state=SEED, stratify=strat[tr_idx])
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1} | train {len(tr2)} | val {len(va2)} | test {len(te_idx)}")
    print(f"{'='*60}")

    tr_loader = make_loader(tr2,   augment=True,  shuffle=True)
    va_loader = make_loader(va2,   augment=False, shuffle=False)
    te_loader = make_loader(te_idx,augment=False, shuffle=False)

    # fresh model each fold
    mamba_f = build_mamba_encoder(MAMBAMIM_PATH, freeze_first_n_blocks=1)
    model_f = E2EFusionModel(mamba_f).to(device)

    # per-group optimiser: Mamba gets lower LR + less weight decay
    # text/struct get higher weight decay to fight overfitting
    opt = torch.optim.AdamW([
        {"params": model_f.cnn.parameters(),
         "lr": LEARNING_RATE*0.1, "weight_decay": 1e-3},   # Mamba: gentle
        {"params": model_f.cbct_enc.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-3},
        {"params": model_f.cbct_head.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-3},
        {"params": model_f.text_enc.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-2},   # text: stronger reg
        {"params": model_f.text_head.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-2},
        {"params": model_f.struct_enc.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-2},
        {"params": model_f.struct_head.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-2},
        {"params": model_f.fusion.parameters(),
         "lr": LEARNING_RATE,     "weight_decay": 1e-2},
    ])
    scaler_amp = GradScaler()

    hist     = {"train_loss":[],"val_loss":[]}
    best_val = -np.inf; ctr=0; best_w=None

    for epoch in range(CV_EPOCHS):
        # ── train ──
        model_f.train(); ep_losses=[]
        for batch in tr_loader:
            vol        = batch["vol"].to(device)
            cbct_pres  = batch["cbct_pres"].to(device)
            fields     = batch["fields"].to(device)
            text_pres  = batch["text_pres"].to(device)
            struct     = batch["struct"].to(device)
            struct_pres= batch["struct_pres"].to(device)
            labels     = batch["label"].to(device)
            opt.zero_grad()
            with autocast():
                out  = model_f(vol,cbct_pres,fields,text_pres,struct,struct_pres)
                loss = (focal_loss(labels,out["logits"]) +
                        AUX_WEIGHT*aux_cbct_loss(labels,out["cbct_aux_logits"],cbct_pres))
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model_f.parameters(),1.0)
            scaler_amp.step(opt); scaler_amp.update()
            ep_losses.append(loss.item())
        hist["train_loss"].append(np.mean(ep_losses))

        # ── validate ──
        model_f.eval(); val_losses=[]
        with torch.no_grad():
            for batch in va_loader:
                vol=batch["vol"].to(device); cbct_pres=batch["cbct_pres"].to(device)
                fields=batch["fields"].to(device); text_pres=batch["text_pres"].to(device)
                struct=batch["struct"].to(device); struct_pres=batch["struct_pres"].to(device)
                labels=batch["label"].to(device)
                with autocast():
                    out=model_f(vol,cbct_pres,fields,text_pres,struct,struct_pres)
                    vl=(focal_loss(labels,out["logits"])+
                        AUX_WEIGHT*aux_cbct_loss(labels,out["cbct_aux_logits"],cbct_pres))
                val_losses.append(vl.item())
        hist["val_loss"].append(np.mean(val_losses))

        vp  = predict_probs_e2e(model_f, va2)
        vpr = macro_pr(all_labels[va2], vp)
        print(f"  E{epoch+1:>3} | train {hist['train_loss'][-1]:.4f} | val {hist['val_loss'][-1]:.4f} | val_PR {vpr:.4f}")

        if vpr>best_val:
            best_val,ctr,best_w=vpr,0,{k:v.cpu().clone() for k,v in model_f.state_dict().items()}
        else:
            ctr+=1
            if ctr>=CV_PATIENCE: print(f"  early stop at epoch {epoch+1}"); break

    if best_w: model_f.load_state_dict({k:v.to(device) for k,v in best_w.items()})
    cv_history.append(hist)

    tp_probs = predict_probs_e2e(model_f, te_idx)
    vp_probs = predict_probs_e2e(model_f, va2)
    yte      = all_labels[te_idx]; yva=all_labels[va2]
    sw,aps,sup = support_weighted_pr(yte, tp_probs)
    fold_sw.append(sw); fold_macro_pr.append(macro_pr(yte,tp_probs))
    fold_macro_roc.append(macro_roc(yte,tp_probs))
    cv_store.append({"y":yte,"probs":tp_probs,"idx":te_idx,"yva":yva,"val_probs":vp_probs})
    ci=0
    for c in range(NUM_CLASSES):
        if 0<yte[:,c].sum()<len(yte): per_class_ap[DISEASE_ORDER[c]].append(aps[ci]); ci+=1
    print(f"\nFold {fold+1}: support-wtd PR {sw:.4f} | macro PR {fold_macro_pr[-1]:.4f} | macro ROC {fold_macro_roc[-1]:.4f}")

    if sw>best_fold_metric:
        best_fold_metric,best_fold_id=sw,fold+1
        torch.save({"model":model_f.state_dict(),"fold":fold+1,"sw_pr":sw}, BEST_PATH)
        best_fold_data={"y":yte,"probs":tp_probs,"idx":te_idx,
                        "yva":yva,"val_probs":vp_probs,"history":hist}
        print(f"  -> new best fold ({sw:.4f}), saved to {BEST_PATH}")

def _ci(v): v=np.array(v); return 1.96*v.std()/np.sqrt(len(v))
print("\n"+"="*62+"\n5-FOLD CV RESULTS (mean +/- std)\n"+"="*62)
print(f"Support-weighted PR-AUC : {np.mean(fold_sw):.4f} +/- {np.std(fold_sw):.4f}  (95% CI {np.mean(fold_sw)-_ci(fold_sw):.4f}-{np.mean(fold_sw)+_ci(fold_sw):.4f})")
print(f"Macro PR-AUC            : {np.mean(fold_macro_pr):.4f} +/- {np.std(fold_macro_pr):.4f}")
print(f"Macro ROC-AUC           : {np.mean(fold_macro_roc):.4f} +/- {np.std(fold_macro_roc):.4f}")
print("\nPer-class PR-AUC (mean +/- std):")
for d in DISEASE_ORDER:
    if per_class_ap[d]: print(f"  {d:<26} {np.mean(per_class_ap[d]):.4f} +/- {np.std(per_class_ap[d]):.4f}")
print(f"\nBEST FOLD: #{best_fold_id} (support-wtd PR-AUC {best_fold_metric:.4f}) -> {BEST_PATH}")


FOLD 1 | train 311 | val 56 | test 92
MambaMIM partial transfer: 0 tensors matched
LightMamba3DEncoder: 30/41 param-tensors trainable
  E  1 | train 0.1102 | val 0.1049 | val_PR 0.4977


## Section 13: Loss Curves (Best Fold)

In [ ]:
# ============================================================
# SECTION 13: BEST FOLD TRAIN/VAL LOSS CURVE
# ============================================================
h  = best_fold_data["history"]
ep = range(1,len(h["train_loss"])+1)
plt.figure(figsize=(9,5))
plt.plot(ep, h["train_loss"],"o-",label="Train loss")
plt.plot(ep, h["val_loss"],  "s-",label="Val loss")
best_ep = int(np.argmin(h["val_loss"]))+1
plt.axvline(best_ep,color="gray",ls="--",alpha=0.6,label=f"Min val loss (ep {best_ep})")
plt.xlabel("Epoch"); plt.ylabel("Focal loss + CBCT aux")
plt.title(f"Best Fold #{best_fold_id} — Train vs Val Loss (End-to-End)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"best_fold_loss_curve.png",dpi=100,bbox_inches="tight"); plt.show()


## Section 14: Load Best-Fold Model

In [ ]:
# ============================================================
# SECTION 14: LOAD BEST-FOLD MODEL
# ============================================================
mamba_best = build_mamba_encoder(MAMBAMIM_PATH, freeze_first_n_blocks=0)
best_model = E2EFusionModel(mamba_best).to(device)
ck         = torch.load(BEST_PATH, map_location=device,
                        weights_only=False)   # ← fix for PyTorch 2.6
best_model.load_state_dict(ck["model"])
best_model.eval()
print(f"Loaded best fold #{ck['fold']} (support-wtd PR-AUC {ck['sw_pr']:.4f})")
print("Mamba encoder — Grad-CAM hooks best_model.cnn.blocks[-1]")

## Section 15: Best-Fold Evaluation

In [ ]:
# ============================================================
# SECTION 15: EVALUATION ON BEST FOLD'S HELD-OUT TEST SET
# ============================================================
bf  = best_fold_data
y,p = bf["y"],bf["probs"]; yv,pv = bf["yva"],bf["val_probs"]
thr = np.full(NUM_CLASSES,0.5)
for c in range(NUM_CLASSES):
    if 0<yv[:,c].sum()<len(yv):
        best=-1
        for t in np.linspace(0.05,0.95,91):
            f=f1_score(yv[:,c],(pv[:,c]>=t).astype(int),zero_division=0)
            if f>best: best,thr[c]=f,t
thresholds=thr.copy()

def report(preds,title,show_thr=False):
    print("\n"+"="*84+f"\nBEST FOLD #{best_fold_id} ({title})\n"+"="*84)
    print(f"{'Disease':<26}"+(f"{'Thr':>7}" if show_thr else "")+f"{'Acc':>8}{'Prec':>8}{'Rec':>8}{'F1':>8}{'ROC':>8}{'PR':>8}")
    for c,d in enumerate(DISEASE_ORDER):
        yt,yp_,pp=y[:,c],preds[:,c],p[:,c]
        acc=accuracy_score(yt,yp_); prec=precision_score(yt,yp_,zero_division=0)
        rec=recall_score(yt,yp_,zero_division=0); f1=f1_score(yt,yp_,zero_division=0)
        roc=roc_auc_score(yt,pp) if 0<yt.sum()<len(yt) else float("nan")
        pr=average_precision_score(yt,pp) if 0<yt.sum()<len(yt) else float("nan")
        tc=f"{thr[c]:>7.2f}" if show_thr else ""
        print(f"{d:<26}{tc}{acc:>8.3f}{prec:>8.3f}{rec:>8.3f}{f1:>8.3f}{roc:>8.3f}{pr:>8.3f}")
report((p>=0.5).astype(int),"@ 0.5")
report((p>=thr).astype(int),"TUNED THRESHOLDS",show_thr=True)
sw_val,_,_=support_weighted_pr(y,p)
print(f"\nBest-fold support-weighted PR-AUC: {sw_val:.4f}")
print(f"(Honest CV mean: {np.mean(fold_sw):.4f} +/- {np.std(fold_sw):.4f})")

# ── balanced accuracy ──
preds_tuned=(p>=thr).astype(int)
for i in range(len(preds_tuned)):
    if preds_tuned[i].sum()==0: preds_tuned[i,np.argmax(p[i])]=1
print("\n"+"="*50+"\nBALANCED ACCURACY\n"+"="*50)
baccs=[]
for i,d in enumerate(DISEASE_ORDER):
    ba=balanced_accuracy_score(y[:,i],preds_tuned[:,i]); baccs.append(ba)
    print(f"  {d:<26}{ba:.3f}")
print(f"  {'MACRO':<26}{np.mean(baccs):.3f}")

# ── modality ablation ──
print("\n"+"="*60+"\nMODALITY ABLATION (macro PR-AUC)\n"+"="*60)
print(f"{'Config':<22}{'Macro ROC':>11}{'Macro PR':>11}")

def ablation_probs(zero_text=False, zero_cbct=False, zero_struct=False):
    """Run inference with one modality zeroed out via presence flag."""
    best_model.eval()
    te_idx = bf["idx"]
    probs  = []
    loader = make_loader(te_idx, augment=False, shuffle=False, batch_size=8)
    with torch.no_grad():
        for batch in loader:
            vol        = batch["vol"].to(device)
            cbct_pres  = batch["cbct_pres"].to(device)
            fields     = batch["fields"].to(device)
            text_pres  = batch["text_pres"].to(device)
            struct     = batch["struct"].to(device)
            struct_pres= batch["struct_pres"].to(device)
            # zero the requested modality's presence flag
            if zero_text:   text_pres   = torch.zeros_like(text_pres)
            if zero_cbct:   cbct_pres   = torch.zeros_like(cbct_pres)
            if zero_struct: struct_pres = torch.zeros_like(struct_pres)
            out=best_model(vol,cbct_pres,fields,text_pres,struct,struct_pres)
            probs.append(torch.sigmoid(out["logits"]).cpu().numpy())
    return np.concatenate(probs,0)

configs = [
    ("All modalities",    False, False, False),
    ("Text only",         False, True,  True),
    ("CBCT only",         True,  False, True),
    ("Structured only",   True,  True,  False),
    ("Text + Structured", False, False, True),
    ("Text + CBCT",       False, True,  False),
]
for name,zt,zc,zs in configs:
    pp=ablation_probs(zero_text=zt, zero_cbct=zc, zero_struct=zs)
    roc=macro_roc(y,pp); pr=macro_pr(y,pp)
    print(f"{name:<22}{roc:>11.3f}{pr:>11.3f}")

# ── fusion weights ──
import scipy.special
W=scipy.special.softmax(best_model.fusion.w.detach().cpu().numpy(),axis=0)
print("\n"+"="*50+"\nFUSION WEIGHTS (Struct / Text / CBCT)\n"+"="*50)
print(f"{'Disease':<26}{'Struct':>8}{'Text':>8}{'CBCT':>8}")
for ci,d in enumerate(DISEASE_ORDER):
    print(f"{d:<26}{W[0,ci]:>8.2f}{W[1,ci]:>8.2f}{W[2,ci]:>8.2f}")

## Section 16: Multi-Label Confusion Matrix (MLCM)

In [ ]:
# ============================================================
# SECTION 16: MLCM — Heydarian et al. (IEEE Access 2022)
# ============================================================
q=NUM_CLASSES
def mlcm_update(M,Ti,Pi):
    Ti,Pi=set(Ti),set(Pi); Ti1=Ti&Pi; Ti2=Ti-Pi; Pi2=Pi-Ti
    for r in Ti1: M[r,r]+=1
    if len(Ti)==0 and len(Pi)==0: M[q,q]+=1
    if Pi.issubset(Ti):
        for r in Ti2: M[r,q]+=1
    elif Ti.issubset(Pi):
        if len(Ti)==0:
            for c in Pi2: M[q,c]+=1
        else:
            for r in Ti:
                for c in Pi2: M[r,c]+=1
    else:
        for r in Ti2:
            for c in Pi2: M[r,c]+=1

preds_m=(p>=thr).astype(int)
M=np.zeros((q+1,q+1),dtype=int)
for i in range(len(y)): mlcm_update(M,list(np.where(y[i]==1)[0]),list(np.where(preds_m[i]==1)[0]))
labels_m=list(DISEASE_ORDER)+["NPL"]; rows_m=list(DISEASE_ORDER)+["NTL"]
print(f"MLCM — best fold #{best_fold_id} ({len(y)} patients)")
print(" "*26+"".join(f"{l[:9]:>11}" for l in labels_m))
for r in range(q+1): print(f"{rows_m[r]:<26}"+"".join(f"{M[r,c]:>11}" for c in range(q+1)))
print("\nPer-class metrics:")
print(f"{'Disease':<26}{'TP':>6}{'FP':>6}{'FN':>6}{'Prec':>8}{'Rec':>8}{'F1':>8}")
for k in range(q):
    TP=M[k,k]; FP=M[:,k].sum()-M[k,k]; FN=M[k,:].sum()-M[k,k]
    prec=TP/(TP+FP) if TP+FP else 0; rec=TP/(TP+FN) if TP+FN else 0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0
    print(f"{DISEASE_ORDER[k]:<26}{TP:>6}{FP:>6}{FN:>6}{prec:>8.3f}{rec:>8.3f}{f1:>8.3f}")
fig,ax=plt.subplots(figsize=(7,6)); ax.imshow(M,cmap="Blues")
ax.set_xticks(range(q+1)); ax.set_xticklabels([l[:10] for l in labels_m],rotation=45,ha="right")
ax.set_yticks(range(q+1)); ax.set_yticklabels([l[:10] for l in rows_m])
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"MLCM — Best Fold #{best_fold_id} (E2E)")
for r in range(q+1):
    for c in range(q+1):
        ax.text(c,r,int(M[r,c]),ha="center",va="center",color="white" if M[r,c]>M.max()/2 else "black")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"mlcm.png",dpi=100,bbox_inches="tight"); plt.show()


## Section 17: FP/FN Reduction — Platt Scaling + Threshold Retuning

In [ ]:
# ============================================================
# SECTION 17: FP/FN REDUCTION
# ============================================================
calibrators=[]; p_cal=np.zeros_like(p)
for c in range(NUM_CLASSES):
    lr_=LogisticRegression(C=1.0,solver="lbfgs",max_iter=1000)
    lr_.fit(pv[:,c:c+1],yv[:,c])
    p_cal[:,c]=lr_.predict_proba(p[:,c:c+1])[:,1]; calibrators.append(lr_)
print("=== CALIBRATION EFFECT ===")
print(f"{'Disease':<26}{'PR before':>11}{'PR after':>11}{'Change':>10}")
for c,d in enumerate(DISEASE_ORDER):
    b=average_precision_score(y[:,c],p[:,c]); a=average_precision_score(y[:,c],p_cal[:,c])
    print(f"{d:<26}{b:>11.3f}{a:>11.3f}{a-b:>+10.3f}")
pv_cal=np.column_stack([calibrators[c].predict_proba(pv[:,c:c+1])[:,1] for c in range(NUM_CLASSES)])
new_thr=np.zeros(NUM_CLASSES)
print("\n=== RETUNED THRESHOLDS ===")
for c,d in enumerate(DISEASE_ORDER):
    beta=1.5 if d=="pulpitis" else 1.0; best=-1
    for t in np.linspace(0.05,0.95,91):
        f=fbeta_score(yv[:,c],(pv_cal[:,c]>=t).astype(int),beta=beta,zero_division=0)
        if f>best: best,new_thr[c]=f,t
    print(f"  {d:<26} {new_thr[c]:.2f}  (F{beta})")
preds_cal=(p_cal>=new_thr).astype(int)
for i in range(len(preds_cal)):
    if preds_cal[i].sum()==0: preds_cal[i,np.argmax(p_cal[i])]=1
print("\n=== BEFORE vs AFTER ===")
preds_old=(p>=thr).astype(int)
for i in range(len(preds_old)):
    if preds_old[i].sum()==0: preds_old[i,np.argmax(p[i])]=1
print(f"{'Disease':<26}{'F1 B':>8}{'F1 A':>8}{'FP B':>7}{'FP A':>7}{'FN B':>7}{'FN A':>7}")
M_new=np.zeros((q+1,q+1),dtype=int)
for i in range(len(y)): mlcm_update(M_new,list(np.where(y[i]==1)[0]),list(np.where(preds_cal[i]==1)[0]))
for k in range(q):
    fb=f1_score(y[:,k],preds_old[:,k],zero_division=0); fa=f1_score(y[:,k],preds_cal[:,k],zero_division=0)
    FPb=M[:,k].sum()-M[k,k]; FNb=M[k,:].sum()-M[k,k]
    FPa=M_new[:,k].sum()-M_new[k,k]; FNa=M_new[k,:].sum()-M_new[k,k]
    print(f"{DISEASE_ORDER[k]:<26}{fb:>8.3f}{fa:>8.3f}{FPb:>7}{FPa:>7}{FNb:>7}{FNa:>7}")


## Section 18: Modality-Level Explainability

In [ ]:
# ============================================================
# SECTION 18: MODALITY EXPLAINABILITY
# (1) Fusion weights  (2) Permutation importance
# ============================================================
import scipy.special
W=scipy.special.softmax(best_model.fusion.w.detach().cpu().numpy(),axis=0)
print("PER-CLASS MODALITY RELIANCE (fusion softmax weights)")
print(f"{'Disease':<26}{'Struct':>8}{'Text':>8}{'CBCT':>8}")
for ci,d in enumerate(DISEASE_ORDER):
    print(f"{d:<26}{W[0,ci]:>8.2f}{W[1,ci]:>8.2f}{W[2,ci]:>8.2f}")

te_idx_full = bf["idx"]   # test patient indices in the full pool
base_probs  = predict_probs_e2e(best_model, te_idx_full)
base_pr     = macro_pr(y, base_probs)
rng         = np.random.default_rng(SEED)
print(f"\nPERMUTATION IMPORTANCE (macro PR-AUC drop when modality shuffled)")
print(f"baseline macro PR-AUC: {base_pr:.4f}")

for mod, feat_name, zero_flag in [
        ("Text",       "field_emb",   "text_pres"),
        ("CBCT",       "cbct_vol",    "cbct_pres"),
        ("Structured", "struct",      "struct_pres")]:
    drops=[]
    for _ in range(10):
        # shuffle by zeroing the presence flag (cleanest permutation for multimodal)
        # For CBCT: zero cbct_present so CNN output is masked in LateFusion
        perm = rng.permutation(len(te_idx_full))
        perm_idx = te_idx_full[perm]   # shuffled patient indices
        # build a mixed dataset: real patients but shuffled modality
        class PermDataset(Dataset):
            def __init__(self, real_idx, perm_idx, mod_name):
                self.real=real_idx; self.perm=perm_idx; self.mod=mod_name
            def __len__(self): return len(self.real)
            def __getitem__(self,i):
                ri=self.real[i]; pi=self.perm[i]
                item={"vol":torch.from_numpy(all_vols[ri].copy()),
                      "fields":torch.from_numpy(all_field_emb[ri]),
                      "text_pres":torch.from_numpy(all_text_pres[ri]),
                      "struct":torch.from_numpy(all_struct[ri]),
                      "struct_pres":torch.from_numpy(all_sp[ri]),
                      "cbct_pres":torch.from_numpy(all_cbct_pres[ri]),
                      "label":torch.from_numpy(all_labels[ri])}
                if self.mod=="Text":
                    item["fields"]=torch.from_numpy(all_field_emb[pi])
                    item["text_pres"]=torch.from_numpy(all_text_pres[pi])
                elif self.mod=="CBCT":
                    item["vol"]=torch.from_numpy(all_vols[pi].copy())
                    item["cbct_pres"]=torch.from_numpy(all_cbct_pres[pi])
                else:
                    item["struct"]=torch.from_numpy(all_struct[pi])
                    item["struct_pres"]=torch.from_numpy(all_sp[pi])
                return item
        pds=PermDataset(te_idx_full,perm_idx,mod)
        pdl=DataLoader(pds,batch_size=8,shuffle=False,num_workers=0)
        best_model.eval(); pp=[]
        with torch.no_grad():
            for batch in pdl:
                out=best_model(batch["vol"].to(device),batch["cbct_pres"].to(device),
                               batch["fields"].to(device),batch["text_pres"].to(device),
                               batch["struct"].to(device),batch["struct_pres"].to(device))
                pp.append(torch.sigmoid(out["logits"]).cpu().numpy())
        pp=np.concatenate(pp,0)
        drops.append(base_pr - macro_pr(y,pp))
    print(f"  {mod:<12} drop {np.mean(drops):+.4f} +/- {np.std(drops):.4f}")


## Section 19: Per-Patient Text Explainability

In [ ]:
# ============================================================
# SECTION 19: TEXT FIELD + WORD ATTRIBUTION
# ============================================================
def get_patient_probs(patient_pool_idx, model):
    """Get probs for one patient by pool index."""
    model.eval()
    with torch.no_grad():
        out=model(
            torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device),
            torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
            torch.from_numpy(all_field_emb[patient_pool_idx:patient_pool_idx+1]).to(device),
            torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
            torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device),
            torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device))
    return torch.sigmoid(out["logits"]).cpu().numpy()[0]

def explain_text_fields(patient_pool_idx):
    pool_y = all_labels[patient_pool_idx]
    base   = get_patient_probs(patient_pool_idx, best_model)
    print(f"--- Patient (pool idx {patient_pool_idx}) ---")
    print("True labels:", [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1] or "(none)")
    print("Predicted probabilities:")
    for c,d in enumerate(DISEASE_ORDER): print(f"   {d:<26} {base[c]:.3f}")
    print("\nField contribution (drop when field removed):")
    print("field".ljust(26)+"".join(d[:10].rjust(12) for d in DISEASE_ORDER))
    for f in range(N_FIELDS):
        fe = all_field_emb[patient_pool_idx:patient_pool_idx+1].copy()
        fe[0,f,:] = 0.0
        best_model.eval()
        with torch.no_grad():
            out=best_model(
                torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(fe).to(device),
                torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device))
        pp=torch.sigmoid(out["logits"]).cpu().numpy()[0]; drop=base-pp
        fname=TEXT_FIELDS[f] if f<N_FIELDS else f"field_{f}"
        print(fname[:26].ljust(26)+"".join(f"{drop[c]:+.3f}".rjust(12) for c in range(NUM_CLASSES)))
    print("(positive = field supported the disease)")

# find a test patient from the best fold's held-out set
te_pool_idx = bf["idx"][0]
explain_text_fields(te_pool_idx)


## Section 20: Combined Text Attribution + Honest E2E Grad-CAM

In [ ]:
# ============================================================
# SECTION 20: COMBINED VISUALIZATION
# Left:  text field attribution bar chart
# Right: E2E Grad-CAM — gradients flow from FUSED logit back
#        through LateFusion → CBCTHead → CBCTEncoder → CNN.layer4
#        → preprocessed voxels. This is HONEST Grad-CAM on the
#        fused prediction, not the standalone CNN.
# ============================================================
DISEASE_COLORS  = {"pulpitis":"Reds","caries":"Blues",
                   "impacted_tooth":"Greens","damaged_or_missing_tooth":"Oranges"}
DISEASE_PCOLORS = {"pulpitis":"red","caries":"dodgerblue",
                   "impacted_tooth":"green","damaged_or_missing_tooth":"orange"}

def e2e_gradcam(patient_pool_idx, disease):
    """
    Honest Grad-CAM: gradient flows from fused_logit[disease]
    back through the entire model to CNN.layer4.
    """
    di     = DISEASE_ORDER.index(disease)
    vol_t  = torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device).float()
    cp_t   = torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device)
    fi_t   = torch.from_numpy(all_field_emb[patient_pool_idx:patient_pool_idx+1]).to(device)
    tp_t   = torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device)
    st_t   = torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device)
    sp_t   = torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device)

    if cp_t.item()<0.5: return None   # no CBCT

    best_model.train()   # need gradients through dropout etc.
    feats={}
    def hook(m,i,o): feats["act"]=o; o.retain_grad()
    h = best_model.cnn.blocks[-1].register_forward_hook(hook)

    out   = best_model(vol_t,cp_t,fi_t,tp_t,st_t,sp_t)
    score = out["logits"][0,di]         # FUSED logit, not CNN-only
    best_model.zero_grad(); score.backward()
    h.remove(); best_model.eval()

    act  = feats["act"]; grad=act.grad  # (1,C,d,h,w)
    w_   = grad.mean(dim=(2,3,4),keepdim=True)
    cam  = F.relu((w_*act).sum(1,keepdim=True))
    cam  = F.interpolate(cam,size=INPUT_SHAPE,mode="trilinear",align_corners=False)
    cam  = cam[0,0].detach().cpu().numpy()
    cam  = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
    return cam

def combined_explain_e2e(patient_pool_idx, slice_axis="axial"):
    pool_y   = all_labels[patient_pool_idx]
    base     = get_patient_probs(patient_pool_idx, best_model)
    predicted= [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if base[c]>=thresholds[c]]
    if not predicted: predicted=[DISEASE_ORDER[int(np.argmax(base))]]
    true_lbl = [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]

    # text field attribution
    field_drops=np.zeros((N_FIELDS,NUM_CLASSES))
    for f in range(N_FIELDS):
        fe=all_field_emb[patient_pool_idx:patient_pool_idx+1].copy(); fe[0,f,:]=0.0
        best_model.eval()
        with torch.no_grad():
            out=best_model(
                torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(fe).to(device),
                torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device))
        field_drops[f]=base-torch.sigmoid(out["logits"]).cpu().numpy()[0]

    # E2E Grad-CAM per predicted disease
    has_cbct = all_cbct_pres[patient_pool_idx,0]>0.5
    cams={}
    if has_cbct:
        for d in predicted:
            cam=e2e_gradcam(patient_pool_idx,d)
            if cam is not None: cams[d]=cam

    vol_np = all_vols[patient_pool_idx,0]   # (96,96,96)

    # pick best slice
    if cams:
        total=sum(cams.values())
        if slice_axis=="axial":   si=int(np.argmax(total.sum(axis=(0,1)))); sl_s=vol_np[:,:,si]; sl_c={d:c[:,:,si] for d,c in cams.items()}; alab=f"Axial z={si}"
        elif slice_axis=="coronal": si=int(np.argmax(total.sum(axis=(0,2)))); sl_s=vol_np[:,si,:]; sl_c={d:c[:,si,:] for d,c in cams.items()}; alab=f"Coronal y={si}"
        else: si=int(np.argmax(total.sum(axis=(1,2)))); sl_s=vol_np[si,:,:]; sl_c={d:c[si,:,:] for d,c in cams.items()}; alab=f"Sagittal x={si}"
    else:
        D=INPUT_SHAPE[2]//2; sl_s=vol_np[:,:,D]; sl_c={}; alab="Axial mid (no CBCT)"

    n_right=2 if has_cbct and cams else 0
    fig=plt.figure(figsize=(7+5*max(n_right,1),7))
    gs=fig.add_gridspec(1,1+max(n_right,1),width_ratios=[2]+[1.8]*max(n_right,1),wspace=0.35)

    # LEFT: text attribution
    ax_t=fig.add_subplot(gs[0,0])
    y_pos=np.arange(N_FIELDS); bw=0.18
    offs=np.linspace(-bw*(NUM_CLASSES-1)/2,bw*(NUM_CLASSES-1)/2,NUM_CLASSES)
    clrs=["#e41a1c","#377eb8","#4daf4a","#ff7f00"]
    for ci,d in enumerate(DISEASE_ORDER):
        ax_t.barh(y_pos+offs[ci],field_drops[:,ci],height=bw,label=d.replace("_"," "),color=clrs[ci],alpha=0.8)
    ax_t.set_yticks(y_pos); ax_t.set_yticklabels([f.replace("_"," ") for f in TEXT_FIELDS],fontsize=9)
    ax_t.axvline(0,color="black",lw=0.8); ax_t.legend(fontsize=8,loc="lower right")
    ax_t.set_xlabel("Prob drop when field removed",fontsize=9); ax_t.grid(axis="x",alpha=0.3)
    ax_t.set_title(f"Text Attribution\nTrue:{true_lbl}\nPredicted:{predicted}",fontsize=9)

    if has_cbct and cams:
        ax_s=fig.add_subplot(gs[0,1])
        ax_s.imshow(sl_s.T,cmap="gray",origin="lower"); ax_s.set_title(f"CBCT {alab}",fontsize=10); ax_s.axis("off")
        ax_o=fig.add_subplot(gs[0,2])
        ax_o.imshow(sl_s.T,cmap="gray",origin="lower")
        for d,cam_sl in sl_c.items():
            msk=np.ma.masked_where(cam_sl.T<0.3,cam_sl.T)
            ax_o.imshow(msk,cmap=DISEASE_COLORS[d],alpha=0.5,origin="lower",vmin=0.3,vmax=1.0)
        patches=[mpatches.Patch(color=DISEASE_PCOLORS[d],label=d.replace("_"," ")) for d in predicted if d in cams]
        ax_o.legend(handles=patches,loc="upper right",fontsize=8,framealpha=0.8)
        ax_o.set_title("E2E Grad-CAM\n(fused logit → CNN.layer4 → voxels)",fontsize=10,color="darkgreen")
        ax_o.axis("off")
    else:
        ax_n=fig.add_subplot(gs[0,1]); ax_n.text(0.5,0.5,"No CBCT\nfor this patient",ha="center",va="center",fontsize=12); ax_n.axis("off")

    plt.suptitle("End-to-End Multimodal Explainability\nGrad-CAM gradients flow from FUSED prediction to voxels (honest)",
                 fontsize=10,color="darkgreen",style="italic",y=1.01)
    fname=OUTPUT_DIR/f"combined_e2e_p{patient_pool_idx}_{slice_axis}.png"
    plt.savefig(fname,dpi=120,bbox_inches="tight"); plt.show()
    print(f"saved {fname}")

# run for all three planes on the first CBCT-present test patient
for pi in bf["idx"]:
    if all_cbct_pres[pi,0]>0.5:
        for axis in ["axial","coronal","sagittal"]:
            combined_explain_e2e(pi, slice_axis=axis)
        break


## Section 21: Treatment Planning

In [ ]:
# ============================================================
# SECTION 21: TREATMENT KNOWLEDGE BASE + QWEN
# ============================================================
from transformers import AutoModelForCausalLM
QWEN_MODEL="Qwen/Qwen2.5-3B-Instruct"
print("loading Qwen...")
qwen_tok=AutoTokenizer.from_pretrained(QWEN_MODEL)
qwen_llm=AutoModelForCausalLM.from_pretrained(QWEN_MODEL,dtype=torch.float16,device_map="auto")
print("Qwen loaded.")


In [ ]:
tx_by_patient=raw_df.groupby("patient_id",sort=False)["treatment_plan"].apply(
    lambda s:" | ".join(t.strip() for t in s.dropna().astype(str) if t.strip()))
patient_df["treatment_plan"]=patient_df["patient_id"].map(tx_by_patient).fillna("")
patient_df["treatment_plan"]=patient_df["treatment_plan"].apply(lambda t:html.unescape(str(t)))
n=(patient_df["treatment_plan"].str.strip().str.len()>0).sum()
print(f"patients with treatment text: {n}/{len(patient_df)}")

def build_treatment_kb(patient_df,label_matrix,disease_order):
    kb=[]
    for i,(_,row) in enumerate(patient_df.iterrows()):
        conditions=[disease_order[c] for c in range(len(disease_order)) if label_matrix[i,c]==1]
        tx=html.unescape(str(row.get("treatment_plan",""))).strip()
        if tx and tx.lower() not in ("nan","none",""):
            kb.append({"patient_id":row["patient_id"],"conditions":conditions,
                       "condition_key":"|".join(sorted(conditions)),"treatment":tx})
    return pd.DataFrame(kb)
kb_df=build_treatment_kb(patient_df,label_matrix,DISEASE_ORDER)
print(f"KB entries: {len(kb_df)}")

URGENCY={"pulpitis":3,"caries":2,"impacted_tooth":1,"damaged_or_missing_tooth":2}
def retrieve_treatments(pc,kb_df,top_k=25):
    pred=set(pc); scored=[]
    for _,r in kb_df.iterrows():
        ov=len(pred&set(r["conditions"]))
        if ov>0: scored.append((ov,set(r["conditions"])==pred,r))
    scored.sort(key=lambda x:(x[0],x[1]),reverse=True)
    return [r for _,_,r in scored[:top_k]]
def dedupe_plans(plans):
    seen,out=set(),[]
    for p in plans:
        k=" ".join(p["treatment"].strip().lower().split())
        if "root canal" in k or k.startswith("rct"):
            k="rct + crown" if "crown" in k else ("rct + filling" if ("resin" in k or "filling" in k) else "rct only")
        elif "extract" in k: k="extraction"
        elif "implant" in k: k="implant"
        elif "filling" in k or "resin" in k: k="filling"
        elif "crown" in k: k="crown"
        if k not in seen: seen.add(k); out.append(p)
    return out
def synthesize_recommendation(pc,kb_df,top_k=8):
    ordered=sorted(pc,key=lambda c:URGENCY.get(c,0),reverse=True)
    ret=retrieve_treatments(pc,kb_df,top_k=max(top_k,25))
    plans=dedupe_plans([{"conditions":r["conditions"],"treatment":r["treatment"]} for r in ret])[:4]
    ptext="\n".join(f"- (conditions: {', '.join(p['conditions'])}) {p['treatment']}" for p in plans)
    prompt=f"""You are a dental clinical decision-support assistant. A patient has been diagnosed with: {', '.join(pc)}.

Below are REAL deduplicated treatment approaches from our clinical dataset:
{ptext}

Rules: Present AT MOST 4 DISTINCT approaches. Group by condition ordered by urgency ({', '.join(ordered)}). Do NOT invent treatments. Do NOT copy specific tooth numbers. End with: "This is decision-support based on {len(plans)} similar cases; final treatment must be determined by a qualified dentist."
Recommendation:"""
    msgs=[{"role":"user","content":prompt}]
    text=qwen_tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=qwen_tok(text,return_tensors="pt").to(qwen_llm.device)
    with torch.no_grad():
        o=qwen_llm.generate(**inp,max_new_tokens=500,temperature=0.3,do_sample=True,top_p=0.9)
    return {"conditions":pc,"recommendation":qwen_tok.decode(o[0][inp["input_ids"].shape[1]:],skip_special_tokens=True).strip()}
print("treatment functions ready")


## Section 22: End-to-End Patient Demo

In [ ]:
# ============================================================
# SECTION 22: END-TO-END DEMO
# ============================================================
def diagnose_and_recommend(patient_pool_idx):
    probs  = get_patient_probs(patient_pool_idx, best_model)
    pool_y = all_labels[patient_pool_idx]
    predicted=[DISEASE_ORDER[c] for c in range(NUM_CLASSES) if probs[c]>=thresholds[c]]
    if not predicted: predicted=[DISEASE_ORDER[int(np.argmax(probs))]]; print("(fallback)")
    actual=[DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]
    print("="*70); print(f"PATIENT (pool idx {patient_pool_idx})"); print("="*70)
    print("Predicted probabilities:")
    for c,d in enumerate(DISEASE_ORDER):
        flag="✓ PREDICTED" if probs[c]>=thresholds[c] else ""
        print(f"  {d:<26} {probs[c]:.3f}  {flag}")
    print(f"\nModel predicted: {predicted}"); print(f"Actual labels:   {actual}")
    print("\n"+"-"*70+"\nTREATMENT RECOMMENDATION:\n"+"-"*70)
    rec=synthesize_recommendation(predicted,kb_df)
    print(rec["recommendation"]); return rec

# demo on 3 test patients from best fold
for pi in bf["idx"][:3]:
    diagnose_and_recommend(pi); print("\n")


## Section 23: Save Artifacts

In [ ]:
# ============================================================
# SECTION 23: SAVE ARTIFACTS FOR DEPLOYMENT
# ============================================================
with open(OUTPUT_DIR/"age_scaler.pkl","wb") as f:
    pickle.dump({"scaler":scaler,"age_mean":age_mean},f)
with open(OUTPUT_DIR/"treatment_kb.pkl","wb") as f:
    pickle.dump(kb_df,f)
config={"DISEASE_ORDER":DISEASE_ORDER,"TEXT_FIELDS":TEXT_FIELDS,
        "BERT_MODEL_NAME":BERT_MODEL_NAME,"NUM_CLASSES":NUM_CLASSES,
        "EMBED_DIM":EMBED_DIM,"HIDDEN_SIZE":HIDDEN_SIZE,"ATTN_DIM":ATTN_DIM,
        "N_FIELDS":N_FIELDS,"INPUT_SHAPE":list(INPUT_SHAPE),
        "best_fold_id":best_fold_id,"cv_sw_mean":float(np.mean(fold_sw)),
        "cv_sw_std":float(np.std(fold_sw))}
with open(OUTPUT_DIR/"config.json","w") as f: json.dump(config,f,indent=2)
print("Saved: age_scaler.pkl, treatment_kb.pkl, config.json, best_e2e_fold.pt")
print("Download these 4 files for the FastAPI service.")


In [ ]:
# ============================================================
# SECTION 20: COMBINED TEXT ATTRIBUTION + HONEST E2E GRAD-CAM
# Left:  text field attribution bar chart
# Right: MIP (Maximum Intensity Projection) of E2E Grad-CAM
#        — three planes projected into one unified view
#        Gradients flow from FUSED logit → CNN.layer4 → voxels
# ============================================================
DISEASE_COLORS  = {"pulpitis":"Reds","caries":"Blues",
                   "impacted_tooth":"Greens","damaged_or_missing_tooth":"Oranges"}
DISEASE_PCOLORS = {"pulpitis":"red","caries":"dodgerblue",
                   "impacted_tooth":"green","damaged_or_missing_tooth":"orange"}

def e2e_gradcam(patient_pool_idx, disease):
    """
    Honest Grad-CAM: gradient flows from fused_logit[disease]
    back through the entire model to CNN.layer4.
    Returns full 3D CAM volume (96,96,96).
    """
    di    = DISEASE_ORDER.index(disease)
    vol_t = torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device).float()
    cp_t  = torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device)
    fi_t  = torch.from_numpy(all_field_emb[patient_pool_idx:patient_pool_idx+1]).to(device)
    tp_t  = torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device)
    st_t  = torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device)
    sp_t  = torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device)

    if cp_t.item() < 0.5: return None

    best_model.train()  # need grad through dropout
    feats = {}
    def hook(m,i,o): feats["act"]=o; o.retain_grad()
    h = best_model.cnn.blocks[-1].register_forward_hook(hook)

    out   = best_model(vol_t,cp_t,fi_t,tp_t,st_t,sp_t)
    score = out["logits"][0, di]   # FUSED logit
    best_model.zero_grad(); score.backward()
    h.remove(); best_model.eval()

    act = feats["act"]; grad = act.grad
    w_  = grad.mean(dim=(2,3,4), keepdim=True)
    cam = F.relu((w_*act).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=INPUT_SHAPE, mode="trilinear", align_corners=False)
    cam = cam[0,0].detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam   # (96,96,96)

def mip_gradcam(cam_3d):
    """
    Maximum Intensity Projection across all three planes.
    Returns three 2D projections: axial, coronal, sagittal.
    """
    mip_axial    = cam_3d.max(axis=2)   # project along z → (96,96)
    mip_coronal  = cam_3d.max(axis=1)   # project along y → (96,96)
    mip_sagittal = cam_3d.max(axis=0)   # project along x → (96,96)
    return mip_axial, mip_coronal, mip_sagittal

def scan_mip(vol_3d):
    """MIP of the scan for background."""
    mip_axial    = vol_3d.max(axis=2)
    mip_coronal  = vol_3d.max(axis=1)
    mip_sagittal = vol_3d.max(axis=0)
    return mip_axial, mip_coronal, mip_sagittal

def combined_explain_e2e(patient_pool_idx):
    pool_y    = all_labels[patient_pool_idx]
    base      = get_patient_probs(patient_pool_idx, best_model)
    predicted = [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if base[c]>=thresholds[c]]
    if not predicted: predicted = [DISEASE_ORDER[int(np.argmax(base))]]
    true_lbl  = [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]

    # ── text field attribution ──
    field_drops = np.zeros((N_FIELDS, NUM_CLASSES))
    for f in range(N_FIELDS):
        fe = all_field_emb[patient_pool_idx:patient_pool_idx+1].copy()
        fe[0,f,:] = 0.0
        best_model.eval()
        with torch.no_grad():
            out = best_model(
                torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(fe).to(device),
                torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device))
        field_drops[f] = base - torch.sigmoid(out["logits"]).cpu().numpy()[0]

    # ── E2E Grad-CAM MIP per predicted disease ──
    has_cbct = all_cbct_pres[patient_pool_idx, 0] > 0.5
    cams_mip = {}   # disease -> (mip_axial, mip_coronal, mip_sagittal)
    if has_cbct:
        for d in predicted:
            cam_3d = e2e_gradcam(patient_pool_idx, d)
            if cam_3d is not None:
                cams_mip[d] = mip_gradcam(cam_3d)

    vol_np = all_vols[patient_pool_idx, 0]   # (96,96,96)
    scan_ax, scan_cor, scan_sag = scan_mip(vol_np)

    # ── FIGURE ──
    # Layout: [text attribution | axial MIP | coronal MIP | sagittal MIP]
    has_cam = has_cbct and len(cams_mip) > 0
    n_cam_panels = 3 if has_cam else 0
    fig = plt.figure(figsize=(6 + 4.5*max(n_cam_panels,1), 7))
    gs  = fig.add_gridspec(1, 1+max(n_cam_panels,1),
                           width_ratios=[2]+[1.5]*max(n_cam_panels,1), wspace=0.3)

    # LEFT: text attribution bar chart
    ax_t = fig.add_subplot(gs[0,0])
    y_pos = np.arange(N_FIELDS); bw = 0.18
    offs  = np.linspace(-bw*(NUM_CLASSES-1)/2, bw*(NUM_CLASSES-1)/2, NUM_CLASSES)
    clrs  = ["#e41a1c","#377eb8","#4daf4a","#ff7f00"]
    for ci,d in enumerate(DISEASE_ORDER):
        ax_t.barh(y_pos+offs[ci], field_drops[:,ci], height=bw,
                  label=d.replace("_"," "), color=clrs[ci], alpha=0.8)
    ax_t.set_yticks(y_pos)
    ax_t.set_yticklabels([f.replace("_"," ") for f in TEXT_FIELDS], fontsize=9)
    ax_t.axvline(0, color="black", lw=0.8)
    ax_t.legend(fontsize=8, loc="lower right")
    ax_t.set_xlabel("Prob drop when field removed", fontsize=9)
    ax_t.grid(axis="x", alpha=0.3)
    ax_t.set_title(f"Text Field Attribution\nTrue: {true_lbl}\nPredicted: {predicted}",
                   fontsize=9)

    # RIGHT: three MIP panels (axial, coronal, sagittal)
    if has_cam:
        mip_scan_views  = [scan_ax,  scan_cor,  scan_sag]
        plane_labels    = ["Axial MIP", "Coronal MIP", "Sagittal MIP"]

        for col, (plane_lbl, sc_mip) in enumerate(zip(plane_labels, mip_scan_views)):
            ax = fig.add_subplot(gs[0, col+1])
            ax.imshow(sc_mip.T, cmap="gray", origin="lower")

            # overlay each disease's MIP CAM
            for d, (mip_ax, mip_cor, mip_sag) in cams_mip.items():
                cam_view = [mip_ax, mip_cor, mip_sag][col]
                masked   = np.ma.masked_where(cam_view.T < 0.3, cam_view.T)
                ax.imshow(masked, cmap=DISEASE_COLORS[d], alpha=0.5,
                          origin="lower", vmin=0.3, vmax=1.0)

            patches = [mpatches.Patch(color=DISEASE_PCOLORS[d],
                                      label=d.replace("_"," "))
                       for d in predicted if d in cams_mip]
            if col == 2:   # legend only on last panel to avoid clutter
                ax.legend(handles=patches, loc="upper right",
                          fontsize=7, framealpha=0.8)
            ax.set_title(plane_lbl, fontsize=10)
            ax.axis("off")
    else:
        ax_n = fig.add_subplot(gs[0,1])
        ax_n.text(0.5, 0.5, "No CBCT\nfor this patient",
                  ha="center", va="center", fontsize=12)
        ax_n.axis("off")

    plt.suptitle(
        "End-to-End Explainability  |  "
        "Grad-CAM: fused logit → CNN.layer4 → voxels (MIP across all planes)",
        fontsize=9, color="darkgreen", style="italic", y=1.01)
    fname = OUTPUT_DIR / f"combined_e2e_mip_p{patient_pool_idx}.png"
    plt.savefig(fname, dpi=120, bbox_inches="tight"); plt.show()
    print(f"saved {fname}")

# run on the first CBCT-present test patient from the best fold
for pi in bf["idx"]:
    if all_cbct_pres[pi, 0] > 0.5:
        combined_explain_e2e(pi)
        break

In [ ]:
# ============================================================
# SECTION 20: COMBINED TEXT ATTRIBUTION + HONEST E2E GRAD-CAM
# Left:  text field attribution bar chart
# Right: best slice per plane (axial, coronal, sagittal)
#        with CAM threshold raised to 0.5 for cleaner hotspots
# ============================================================
DISEASE_COLORS  = {"pulpitis":"Reds","caries":"Blues",
                   "impacted_tooth":"Greens","damaged_or_missing_tooth":"Oranges"}
DISEASE_PCOLORS = {"pulpitis":"red","caries":"dodgerblue",
                   "impacted_tooth":"green","damaged_or_missing_tooth":"orange"}

def e2e_gradcam(patient_pool_idx, disease):
    di    = DISEASE_ORDER.index(disease)
    vol_t = torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device).float()
    cp_t  = torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device)
    fi_t  = torch.from_numpy(all_field_emb[patient_pool_idx:patient_pool_idx+1]).to(device)
    tp_t  = torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device)
    st_t  = torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device)
    sp_t  = torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device)
    if cp_t.item()<0.5: return None
    best_model.train()
    feats={}
    def hook(m,i,o): feats["act"]=o; o.retain_grad()
    h = best_model.cnn.blocks[-1].register_forward_hook(hook)
    out  =best_model(vol_t,cp_t,fi_t,tp_t,st_t,sp_t)
    score=out["logits"][0,di]
    best_model.zero_grad(); score.backward()
    h.remove(); best_model.eval()
    act=feats["act"]; grad=act.grad
    w_ =grad.mean(dim=(2,3,4),keepdim=True)
    cam=F.relu((w_*act).sum(1,keepdim=True))
    cam=F.interpolate(cam,size=INPUT_SHAPE,mode="trilinear",align_corners=False)
    cam=cam[0,0].detach().cpu().numpy()
    cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
    return cam

def best_slice(vol_3d, cam_total):
    """Pick the single richest slice per plane based on total CAM."""
    bz = int(np.argmax(cam_total.sum(axis=(0,1))))  # axial
    by = int(np.argmax(cam_total.sum(axis=(0,2))))  # coronal
    bx = int(np.argmax(cam_total.sum(axis=(1,2))))  # sagittal
    return (
        (vol_3d[:,:,bz], bz, "Axial"),
        (vol_3d[:,by,:], by, "Coronal"),
        (vol_3d[bx,:,:], bx, "Sagittal"),
    )

def combined_explain_e2e(patient_pool_idx, cam_threshold=0.45):
    pool_y   = all_labels[patient_pool_idx]
    base     = get_patient_probs(patient_pool_idx, best_model)
    predicted= [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if base[c]>=thresholds[c]]
    if not predicted: predicted=[DISEASE_ORDER[int(np.argmax(base))]]
    true_lbl = [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]

    # ── text field attribution ──
    field_drops=np.zeros((N_FIELDS,NUM_CLASSES))
    for f in range(N_FIELDS):
        fe=all_field_emb[patient_pool_idx:patient_pool_idx+1].copy(); fe[0,f,:]=0.0
        best_model.eval()
        with torch.no_grad():
            out=best_model(
                torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(fe).to(device),
                torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device))
        field_drops[f]=base-torch.sigmoid(out["logits"]).cpu().numpy()[0]

    # ── E2E Grad-CAM per predicted disease ──
    has_cbct=all_cbct_pres[patient_pool_idx,0]>0.5
    cams_3d={}
    if has_cbct:
        for d in predicted:
            cam=e2e_gradcam(patient_pool_idx,d)
            if cam is not None: cams_3d[d]=cam

    vol_np=all_vols[patient_pool_idx,0]

    # ── FIGURE ──
    has_cam=has_cbct and len(cams_3d)>0
    fig=plt.figure(figsize=(18,6))

    if has_cam:
        # 4 panels: text | axial | coronal | sagittal
        gs=fig.add_gridspec(1,4,width_ratios=[2.5,1.8,1.8,1.8],wspace=0.25)
        cam_total=sum(cams_3d.values())
        slice_views=best_slice(vol_np,cam_total)
    else:
        gs=fig.add_gridspec(1,2,width_ratios=[2.5,1.8],wspace=0.25)

    # LEFT: text attribution
    ax_t=fig.add_subplot(gs[0,0])
    y_pos=np.arange(N_FIELDS); bw=0.15
    offs=np.linspace(-bw*(NUM_CLASSES-1)/2,bw*(NUM_CLASSES-1)/2,NUM_CLASSES)
    clrs=["#e41a1c","#377eb8","#4daf4a","#ff7f00"]
    max_drop=0
    for ci,d in enumerate(DISEASE_ORDER):
        vals=field_drops[:,ci]
        ax_t.barh(y_pos+offs[ci],vals,height=bw,
                  label=d.replace("_"," "),color=clrs[ci],alpha=0.85)
        max_drop=max(max_drop,abs(vals).max())
    ax_t.set_yticks(y_pos)
    ax_t.set_yticklabels([f.replace("_"," ") for f in TEXT_FIELDS],fontsize=10)
    ax_t.axvline(0,color="black",lw=0.8)
    # set x-axis to actual data range so bars are visible
    pad=max_drop*0.15 if max_drop>0 else 0.05
    ax_t.set_xlim(-max_drop-pad, max_drop+pad)
    ax_t.legend(fontsize=8,loc="lower right",framealpha=0.8)
    ax_t.set_xlabel("Prob drop when field removed\n(positive = field supported disease)",fontsize=9)
    ax_t.grid(axis="x",alpha=0.3)
    ax_t.set_title(
        f"Text Field Attribution\n"
        f"True: {[d.replace('_',' ') for d in true_lbl]}\n"
        f"Predicted: {[d.replace('_',' ') for d in predicted]}",
        fontsize=9)

    # RIGHT: best slice per plane with CAM overlay
    if has_cam:
        for col,(sl_scan,si,plane_name) in enumerate(slice_views):
            ax=fig.add_subplot(gs[0,col+1])
            ax.imshow(sl_scan.T,cmap="gray",origin="lower")
            for d,cam_3d in cams_3d.items():
                # pick the same slice index for this plane
                cam_sl = [cam_3d[:,:,si], cam_3d[:,si,:], cam_3d[si,:,:]][col]
                # only show regions above threshold — keeps hotspots clean
                masked=np.ma.masked_where(cam_sl.T<cam_threshold,cam_sl.T)
                ax.imshow(masked,cmap=DISEASE_COLORS[d],alpha=0.55,
                          origin="lower",vmin=cam_threshold,vmax=1.0)
            ax.set_title(f"{plane_name} (slice {si})",fontsize=10)
            ax.axis("off")
            # legend on first CAM panel only
            if col==0:
                patches=[mpatches.Patch(color=DISEASE_PCOLORS[d],
                                        label=d.replace("_"," "))
                         for d in predicted if d in cams_3d]
                ax.legend(handles=patches,loc="upper right",fontsize=8,framealpha=0.85)
    else:
        ax_n=fig.add_subplot(gs[0,1])
        ax_n.text(0.5,0.5,"No CBCT\nfor this patient",
                  ha="center",va="center",fontsize=12); ax_n.axis("off")

    plt.suptitle(
        "End-to-End Explainability  |  "
        "Grad-CAM: fused logit → CNN.layer4 → voxels  |  "
        f"CAM threshold: {cam_threshold}",
        fontsize=9,color="darkgreen",style="italic",y=1.02)
    fname=OUTPUT_DIR/f"combined_e2e_p{patient_pool_idx}.png"
    plt.savefig(fname,dpi=130,bbox_inches="tight"); plt.show()
    print(f"saved {fname}")

# run on first CBCT-present test patient from best fold
for pi in bf["idx"]:
    if all_cbct_pres[pi,0]>0.5:
        combined_explain_e2e(pi, cam_threshold=0.45)
        break

In [ ]:
# try combined explainability on multiple CBCT-present test patients
cbct_test_patients = [pi for pi in bf["idx"] if all_cbct_pres[pi,0]>0.5]
print(f"CBCT-present patients in best fold test set: {len(cbct_test_patients)}")
print(f"Pool indices: {cbct_test_patients[:10]}")  # show first 10

# run on first 5 CBCT-present patients
for pi in cbct_test_patients[:5]:
    pool_y   = all_labels[pi]
    base     = get_patient_probs(pi, best_model)
    predicted= [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if base[c]>=thresholds[c]]
    if not predicted: predicted=[DISEASE_ORDER[int(np.argmax(base))]]
    true_lbl = [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]
    print(f"\nPatient pool idx {pi} | true: {true_lbl} | predicted: {predicted} | probs: {[f'{base[c]:.2f}' for c in range(NUM_CLASSES)]}")
    combined_explain_e2e(pi, cam_threshold=0.45)

In [ ]:
# ============================================================
# COMBINED TEXT ATTRIBUTION + E2E GRAD-CAM (MIP version)
# Higher threshold + better contrast to avoid colour wash
# ============================================================
def combined_explain_mip(patient_pool_idx, cam_threshold=0.55, alpha=0.6):
    pool_y   = all_labels[patient_pool_idx]
    base     = get_patient_probs(patient_pool_idx, best_model)
    predicted= [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if base[c]>=thresholds[c]]
    if not predicted: predicted=[DISEASE_ORDER[int(np.argmax(base))]]
    true_lbl = [DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]

    # ── text field attribution ──
    field_drops=np.zeros((N_FIELDS,NUM_CLASSES))
    for f in range(N_FIELDS):
        fe=all_field_emb[patient_pool_idx:patient_pool_idx+1].copy(); fe[0,f,:]=0.0
        best_model.eval()
        with torch.no_grad():
            out=best_model(
                torch.from_numpy(all_vols[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_cbct_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(fe).to(device),
                torch.from_numpy(all_text_pres[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_struct[patient_pool_idx:patient_pool_idx+1]).to(device),
                torch.from_numpy(all_sp[patient_pool_idx:patient_pool_idx+1]).to(device))
        field_drops[f]=base-torch.sigmoid(out["logits"]).cpu().numpy()[0]

    # ── E2E Grad-CAM per predicted disease ──
    has_cbct=all_cbct_pres[patient_pool_idx,0]>0.5
    cams_3d={}
    if has_cbct:
        for d in predicted:
            cam=e2e_gradcam(patient_pool_idx,d)
            if cam is not None: cams_3d[d]=cam

    vol_np=all_vols[patient_pool_idx,0]  # (96,96,96)

    # ── MIP of scan and CAMs ──
    # use mean-of-top-30% MIP instead of pure max — reduces wash-out
    def smart_mip(vol3d, axis, top_frac=0.3):
        """Average the top fraction of slices by CAM activity."""
        if axis==0:   proj=vol3d
        elif axis==1: proj=vol3d.transpose(1,0,2)
        else:         proj=vol3d.transpose(2,0,1)
        n_keep=max(1,int(proj.shape[0]*top_frac))
        activity=proj.sum(axis=(1,2))
        top_idx=np.argsort(activity)[-n_keep:]
        return proj[top_idx].max(axis=0)

    # scan MIP — plain max looks fine for the grayscale background
    sc_ax  = vol_np.max(axis=2)    # (96,96)
    sc_cor = vol_np.max(axis=1)    # (96,96)
    sc_sag = vol_np.max(axis=0)    # (96,96)

    # CAM smart-MIP — average top 30% of activated slices per disease
    cam_views={}  # disease -> (ax,cor,sag) 2D arrays
    for d,cam3d in cams_3d.items():
        cam_views[d]=(
            smart_mip(cam3d,2),   # axial
            smart_mip(cam3d,1),   # coronal
            smart_mip(cam3d,0),   # sagittal
        )

    # ── FIGURE ──
    has_cam=has_cbct and len(cam_views)>0
    fig=plt.figure(figsize=(18,6))
    if has_cam:
        gs=fig.add_gridspec(1,4,width_ratios=[2.5,1.8,1.8,1.8],wspace=0.25)
    else:
        gs=fig.add_gridspec(1,2,width_ratios=[2.5,1.8],wspace=0.25)

    # LEFT: text attribution
    ax_t=fig.add_subplot(gs[0,0])
    y_pos=np.arange(N_FIELDS); bw=0.15
    offs=np.linspace(-bw*(NUM_CLASSES-1)/2,bw*(NUM_CLASSES-1)/2,NUM_CLASSES)
    clrs=["#e41a1c","#377eb8","#4daf4a","#ff7f00"]
    max_drop=max(abs(field_drops).max(),0.01)
    for ci,d in enumerate(DISEASE_ORDER):
        ax_t.barh(y_pos+offs[ci],field_drops[:,ci],height=bw,
                  label=d.replace("_"," "),color=clrs[ci],alpha=0.85)
    ax_t.set_yticks(y_pos)
    ax_t.set_yticklabels([f.replace("_"," ") for f in TEXT_FIELDS],fontsize=10)
    ax_t.axvline(0,color="black",lw=0.8)
    pad=max_drop*0.15
    ax_t.set_xlim(-max_drop-pad,max_drop+pad)
    ax_t.legend(fontsize=8,loc="lower right",framealpha=0.8)
    ax_t.set_xlabel("Prob drop when field removed\n(positive = field supported disease)",fontsize=9)
    ax_t.grid(axis="x",alpha=0.3)
    ax_t.set_title(
        f"Text Field Attribution\n"
        f"True: {[d.replace('_',' ') for d in true_lbl]}\n"
        f"Predicted: {[d.replace('_',' ') for d in predicted]}",
        fontsize=9)

    # RIGHT: three MIP panels
    if has_cam:
        scan_views=[sc_ax, sc_cor, sc_sag]
        plane_names=["Axial MIP","Coronal MIP","Sagittal MIP"]

        for col,(sc,plane) in enumerate(zip(scan_views,plane_names)):
            ax=fig.add_subplot(gs[0,col+1])
            # normalise scan MIP for better contrast
            sc_norm=(sc-sc.min())/(sc.max()-sc.min()+1e-8)
            ax.imshow(sc_norm.T,cmap="gray",origin="lower",vmin=0,vmax=1)
            for d,(cam_ax,cam_cor,cam_sag) in cam_views.items():
                cam_2d=[cam_ax,cam_cor,cam_sag][col]
                # normalise per-disease CAM independently
                cam_2d=(cam_2d-cam_2d.min())/(cam_2d.max()-cam_2d.min()+1e-8)
                masked=np.ma.masked_where(cam_2d.T<cam_threshold,cam_2d.T)
                ax.imshow(masked,cmap=DISEASE_COLORS[d],alpha=alpha,
                          origin="lower",vmin=cam_threshold,vmax=1.0)
            ax.set_title(plane,fontsize=10); ax.axis("off")
            if col==0:
                patches=[mpatches.Patch(color=DISEASE_PCOLORS[d],
                                        label=d.replace("_"," "))
                         for d in predicted if d in cam_views]
                ax.legend(handles=patches,loc="upper right",fontsize=8,framealpha=0.85)
    else:
        ax_n=fig.add_subplot(gs[0,1])
        ax_n.text(0.5,0.5,"No CBCT\nfor this patient",
                  ha="center",va="center",fontsize=12); ax_n.axis("off")

    plt.suptitle(
        f"End-to-End Explainability  |  Patient pool idx {patient_pool_idx}  |  "
        f"Grad-CAM: fused logit → CNN.layer4 → voxels (smart MIP, threshold {cam_threshold})",
        fontsize=8,color="darkgreen",style="italic",y=1.02)
    fname=OUTPUT_DIR/f"combined_e2e_smip_p{patient_pool_idx}.png"
    plt.savefig(fname,dpi=130,bbox_inches="tight"); plt.show()
    print(f"saved {fname}")

# run on first 5 CBCT-present test patients
cbct_test_patients=[pi for pi in bf["idx"] if all_cbct_pres[pi,0]>0.5]
print(f"CBCT-present in best fold test: {len(cbct_test_patients)}")
for pi in cbct_test_patients[:5]:
    pool_y  =all_labels[pi]
    base    =get_patient_probs(pi,best_model)
    predicted=[DISEASE_ORDER[c] for c in range(NUM_CLASSES) if base[c]>=thresholds[c]]
    if not predicted: predicted=[DISEASE_ORDER[int(np.argmax(base))]]
    true_lbl=[DISEASE_ORDER[c] for c in range(NUM_CLASSES) if pool_y[c]==1]
    print(f"\nPool idx {pi} | true: {true_lbl} | predicted: {predicted}")
    combined_explain_mip(pi, cam_threshold=0.55, alpha=0.6)